# Chapter 10 — Problem Set 1: Solutions

Runnable solutions with explanations. Alternative approaches are noted where relevant.

---
*Generated by Berta AI*

## 1. Tokenization Challenges — Solution

In [ ]:
import nltk
nltk.download('punkt', quiet=True)
from nltk import word_tokenize

s = "I'm sure we've seen it—don't you think?"
tokens_nltk = word_tokenize(s)
tokens_split = s.split()
print("NLTK:", tokens_nltk)
print("split():", tokens_split)
print("NLTK splits contractions and punctuation; split() keeps punctuation attached.")

## 2. Text Cleaning — Solution

In [ ]:
import re

def clean_messy(text):
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = ' '.join(text.lower().split())
    return text

messy = "  Check https://example.com  and @user   !!  "
print("Before:", repr(messy))
print("After:", repr(clean_messy(messy)))

## 3. TF-IDF from Scratch — Solution

In [ ]:
import numpy as np
from collections import Counter
from nltk import word_tokenize

docs = ["the cat sat", "the dog sat", "cat and dog"]
tokenized = [word_tokenize(d.lower()) for d in docs]
vocab = sorted(set(w for t in tokenized for w in t))
N = len(docs)
idf = {w: np.log((N + 1) / (1 + sum(1 for t in tokenized if w in t))) for w in vocab}

def tfidf_vec(tokens):
    tf = Counter(tokens)
    return [tf.get(w, 0) * idf[w] for w in vocab]

X_scratch = np.array([tfidf_vec(t) for t in tokenized])
print("From scratch (vocab):", vocab)
print(X_scratch)

from sklearn.feature_extraction.text import TfidfVectorizer
vec = TfidfVectorizer()
X_sk = vec.fit_transform(docs).toarray()
print("Sklearn (uses L2 norm by default):")
print(vec.get_feature_names_out())
print(X_sk)

## 4. Word Similarity — Solution

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..', 'scripts'))
from embedding_utils import find_similar_words, word_analogy

# Demo with tiny vocab (use real GloVe path if available)
import numpy as np
vocab = "good great bad computer run running runner king queen man woman paris france italy rome".split()
np.random.seed(42)
emb = {w: np.random.randn(50).astype(np.float32) for w in vocab}
emb["great"] = emb["good"] + 0.2 * np.random.randn(50).astype(np.float32)
emb["queen"] = emb["king"] - emb["man"] + emb["woman"] + 0.1 * np.random.randn(50).astype(np.float32)

for w in ["good", "computer", "run"]:
    print(f"Similar to '{w}':", find_similar_words(w, emb, n=3))
print("king - man + woman:", word_analogy("king", "man", "woman", emb, n=3))
print("paris - france + italy:", word_analogy("paris", "france", "italy", emb, n=3))

## 5. Simple Sentiment — Solution

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

path = os.path.join('..', '..', 'datasets', 'reviews.csv')
if os.path.exists(path):
    df = pd.read_csv(path)
else:
    df = pd.DataFrame({'text': ["Great!", "Bad.", "Loved it.", "Hated it."], 'sentiment': [1,0,1,0]})

X = df['text'].fillna('').values
y = df['sentiment'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

tfidf = TfidfVectorizer(max_features=5000)
X_tr = tfidf.fit_transform(X_train)
X_te = tfidf.transform(X_test)
clf = LogisticRegression(max_iter=500)
clf.fit(X_tr, y_train)
pred = clf.predict(X_te)
print("Accuracy:", accuracy_score(y_test, pred))
print("F1:", f1_score(y_test, pred))
print(classification_report(y_test, pred))
print(confusion_matrix(y_test, pred))

## 6. Vocabulary Building — Solution

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..', 'scripts'))
from text_preprocessing import build_vocabulary

texts = ["the cat sat on the mat", "the dog sat", "cat and dog"]
vocab = build_vocabulary(texts, min_frequency=1)
print("Vocabulary (word -> index):", vocab)
print("Size:", len(vocab))
held_out = "the unknown word zebra"
tokens = word_tokenize(held_out.lower())
oov = sum(1 for t in tokens if t not in vocab)
print("OOV fraction on held-out:", oov / len(tokens) if tokens else 0)

## 7. BoW vs TF-IDF Similarity — Solution

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

docs = ["machine learning is great", "machine learning and AI", "weather is great today"]
cv = CountVectorizer()
tfidf = TfidfVectorizer()
bow = cv.fit_transform(docs)
tf = tfidf.fit_transform(docs)
print("Cosine sim doc0 vs doc1 — BoW:", cosine_similarity(bow[0:1], bow[1:2])[0,0])
print("Cosine sim doc0 vs doc1 — TF-IDF:", cosine_similarity(tf[0:1], tf[1:2])[0,0])
print("TF-IDF downweights common terms, so similarity can differ.")